# VGG16 Feature Extractor + SVM — Göz Bölgesi Deepfake Deneyi (FIXED)

Bu notebook, kaş bölgesi için hazırlanmış **VGG16 Feature Extractor + SVM** deneyini
**Kader → Deney 1 → Göz → eye_roi_output** veri yapısına uyarlanmış biçimde çalıştırır.

**Model girdisi**

`eye_roi_output/{real,fake}/{train,val,test}/combined/**`

Ana veri kaynağı `eye_roi_output/metadata.csv` dosyasıdır. Yalnızca `status=success`
olan ve geçerli `combined_eye_path` dosyasına sahip örnekler kullanılır.

**Pozitif sınıf:** `FAKE = 1`  
**Negatif sınıf:** `REAL = 0`

**Çıktı**

Tüm sonuçlar yalnızca:

`Kader / Deney 1 / Sonuçlar / VGG16_FeatureExtractor_SVM_Goz/`

altına yazılır. Göz ROI kaynak dosyaları **silinmez, taşınmaz, yeniden adlandırılmaz ve üzerine yazılmaz**.

Pipeline:

`Combined Eye ROI → VGG16 (ImageNet, include_top=False, GAP) → 512-D feature → StandardScaler → SVM → REAL/FAKE`

> Not: Mevcut `train / val / test` ayrımı aynen korunur. Notebook yeni split üretmez.


In [1]:
# ============================================================
# 0) COLAB / DEPENDENCY SETUP
# ============================================================

!pip -q install "scikit-learn>=1.4,<2" "PyYAML>=6,<7" joblib tqdm

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["PYTHONHASHSEED"] = "42"

import sys
import json
import time
import random
import hashlib
import logging
import platform
import subprocess
import re
import unicodedata
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import joblib
from tqdm.auto import tqdm
from PIL import Image, UnidentifiedImageError

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Python      :", sys.version.split()[0])
print("TensorFlow :", tf.__version__)
print("GPU         :", tf.config.list_physical_devices("GPU"))
print("Seed        :", SEED)


Python      : 3.12.13
TensorFlow : 2.20.0
GPU         : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Seed        : 42


In [2]:
# ============================================================
# 1) DRIVE + GÖZ VERİ YOLLARI + CONFIG + LOGGING
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

MYDRIVE = Path("/content/drive/MyDrive")
SHARED_DRIVES = Path("/content/drive/Shareddrives")

def _norm_name(value: str) -> str:
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return value.casefold().strip()

def find_unique_child(parent: Path, accepted_names):
    if not parent.exists():
        raise FileNotFoundError(f"Parent directory does not exist: {parent}")

    wanted = {_norm_name(x) for x in accepted_names}
    matches = [
        p for p in parent.iterdir()
        if p.is_dir() and _norm_name(p.name) in wanted
    ]

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        existing = [p.name for p in parent.iterdir() if p.is_dir()]
        raise FileNotFoundError(
            f"Expected one of {accepted_names} under {parent}.\n"
            f"Existing directories: {existing[:60]}"
        )

    raise RuntimeError(f"Ambiguous directory match under {parent}: {matches}")

def resolve_project_root():
    accepted_aisc_names = [
        "AISC DeepFake Çalışmaları",
        "AISC Deepfake Çalışmaları",
        "AISC Çalışmalar",
        "AISC Çalışmalar",
    ]

    # Önce MyDrive altında güvenli ve hızlı çözüm.
    if MYDRIVE.exists():
        try:
            return find_unique_child(MYDRIVE, accepted_aisc_names)
        except FileNotFoundError:
            pass

    # Gerekirse Shared drives altında aynı isimleri ara.
    matches = []
    if SHARED_DRIVES.exists():
        wanted = {_norm_name(x) for x in accepted_aisc_names}
        for current_root, dirs, _ in os.walk(SHARED_DRIVES):
            current = Path(current_root)
            dirs[:] = [d for d in dirs if not d.startswith(".")]
            if _norm_name(current.name) in wanted:
                matches.append(current)
                dirs[:] = []

    unique = []
    seen = set()
    for p in matches:
        key = str(p)
        if key not in seen:
            unique.append(p)
            seen.add(key)

    if len(unique) == 1:
        return unique[0]
    if not unique:
        raise FileNotFoundError(
            "AISC DeepFake Çalışmaları klasörü MyDrive veya Shared drives altında bulunamadı."
        )
    raise RuntimeError(f"Birden fazla AISC proje kökü bulundu: {unique}")

AISC_ROOT = resolve_project_root()
DENEYLER_ROOT = find_unique_child(AISC_ROOT, ["Deneyler"])
KADER_ROOT = find_unique_child(DENEYLER_ROOT, ["Kader"])
DENEY1_ROOT = find_unique_child(KADER_ROOT, ["Deney 1", "Deney1"])
EYE_ROOT = find_unique_child(DENEY1_ROOT, ["Göz", "Goz"])
EYE_ROI_ROOT = find_unique_child(EYE_ROOT, ["eye_roi_output"])
RESULTS_ROOT = find_unique_child(DENEY1_ROOT, ["Sonuçlar", "Sonuclar"])

METADATA_CSV = EYE_ROI_ROOT / "metadata.csv"
if not METADATA_CSV.is_file():
    raise FileNotFoundError(f"Eye ROI metadata bulunamadı: {METADATA_CSV}")

RUN_NAME = "VGG16_FeatureExtractor_SVM_Goz"
OUTPUT_DIR = RESULTS_ROOT / RUN_NAME

CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
LOGS_DIR = OUTPUT_DIR / "logs"
METRICS_DIR = OUTPUT_DIR / "metrics"
PREDICTIONS_DIR = OUTPUT_DIR / "predictions"
FIGURES_DIR = OUTPUT_DIR / "figures"
ARTIFACTS_DIR = OUTPUT_DIR / "artifacts"

for d in [
    OUTPUT_DIR,
    CHECKPOINTS_DIR,
    LOGS_DIR,
    METRICS_DIR,
    PREDICTIONS_DIR,
    FIGURES_DIR,
    ARTIFACTS_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "experiment_name": RUN_NAME,
    "region": "eye",
    "roi_variant": "combined",
    "dataset_dir": str(EYE_ROI_ROOT),
    "metadata_csv": str(METADATA_CSV),
    "output_dir": str(OUTPUT_DIR),
    "labels": {"real": 0, "fake": 1},
    "splits": ["train", "val", "test"],
    "image_size": [224, 224],
    "batch_size": 32,
    "seed": SEED,
    "feature_extractor": {
        "architecture": "VGG16",
        "weights": "imagenet",
        "include_top": False,
        "pooling": "avg",
        "trainable": False,
        "feature_dim": 512,
    },
    "svm": {
        "class_weight": "balanced",
        "selection_metric": "f1",
        "search_space": {
            "linear_C": [0.1, 1, 10],
            "rbf_C": [1, 10, 100],
            "rbf_gamma": ["scale", 0.001, 0.01],
        },
    },
    "reuse_cached_features": True,
}

LOG_FILE = LOGS_DIR / "training.log"

logger = logging.getLogger("vgg16_svm_eye")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False

# Her çalışmada logu temiz başlatır; model/veri dosyalarına dokunmaz.
file_handler = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")
stream_handler = logging.StreamHandler(sys.stdout)

formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

with open(OUTPUT_DIR / "config_resolved.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, allow_unicode=True, sort_keys=False)

print("\n✅ DRIVE VE GÖZ DİZİN KURULUMU TAMAM")
print("DENEY1_ROOT  :", DENEY1_ROOT)
print("EYE_ROI_ROOT :", EYE_ROI_ROOT)
print("METADATA_CSV :", METADATA_CSV)
print("OUTPUT_DIR   :", OUTPUT_DIR)

logger.info("Experiment initialized.")
logger.info("Eye ROI root: %s", EYE_ROI_ROOT)
logger.info("Metadata    : %s", METADATA_CSV)
logger.info("Output      : %s", OUTPUT_DIR)


Mounted at /content/drive

✅ DRIVE VE GÖZ DİZİN KURULUMU TAMAM
DENEY1_ROOT  : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1
EYE_ROI_ROOT : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output
METADATA_CSV : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/metadata.csv
OUTPUT_DIR   : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/VGG16_FeatureExtractor_SVM_Goz
2026-08-08 09:59:57 | INFO | Experiment initialized.
2026-08-08 09:59:57 | INFO | Eye ROI root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output
2026-08-08 09:59:57 | INFO | Metadata    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/metadata.csv
2026-08-08 09:59:57 | INFO | Output      : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/VGG16_FeatureExtract

### Veri kaynağı göz ROI metadata'sıdır

Bu sürüm kaş klasörlerini taramak yerine `Göz/eye_roi_output/metadata.csv` dosyasını
tek doğruluk kaynağı olarak kullanır. Yalnızca başarılı **combined-eye ROI** kayıtları
eğitime alınır.


### Split güvenliği

Notebook metadata'daki mevcut `train / val / test` değerlerini aynen korur.
Ayrıca aynı görüntü içeriğinin SHA-256 özeti farklı splitlerde görünürse deney
otomatik olarak durdurulur.


In [ ]:
# ============================================================
# 4) EYE ROI METADATA + VERİ ENVANTERİ
# ============================================================

REQUIRED_COLUMNS = {
    "label",
    "split",
    "combined_eye_path",
    "status",
}

metadata = pd.read_csv(METADATA_CSV)

missing_columns = REQUIRED_COLUMNS.difference(metadata.columns)
if missing_columns:
    raise ValueError(
        f"Eye ROI metadata eksik zorunlu sütunlar içeriyor: {sorted(missing_columns)}"
    )

metadata = metadata.copy()
metadata["label"] = metadata["label"].astype(str).str.lower().str.strip()
metadata["split"] = metadata["split"].astype(str).str.lower().str.strip()
metadata["status"] = metadata["status"].astype(str).str.lower().str.strip()

# val / validation adlandırmalarını tek biçime getir.
metadata["split"] = metadata["split"].replace({
    "validation": "val",
    "valid": "val",
})

allowed_labels = set(CONFIG["labels"])
allowed_splits = set(CONFIG["splits"])

bad_labels = sorted(set(metadata["label"].dropna()) - allowed_labels)
bad_splits = sorted(set(metadata["split"].dropna()) - allowed_splits)

if bad_labels:
    raise ValueError(f"Beklenmeyen label değerleri: {bad_labels}")
if bad_splits:
    raise ValueError(f"Beklenmeyen split değerleri: {bad_splits}")

def is_success_status(value: str) -> bool:
    return str(value).strip().casefold() in {"success", "ok", "completed", "complete"}

def resolve_combined_eye_path(raw_path, label, split):
    raw = "" if pd.isna(raw_path) else str(raw_path).strip()
    if not raw:
        return None

    p = Path(raw)

    candidates = []

    # 1) Metadata içinde halen geçerli mutlak yol varsa.
    if p.is_absolute():
        candidates.append(p)

    # 2) eye_roi_output köküne göre relative yol.
    candidates.append(EYE_ROI_ROOT / raw)

    # 3) Deney 1 köküne göre relative yol.
    candidates.append(DENEY1_ROOT / raw)

    # 4) Eski Colab/Drive mutlak yolunda "eye_roi_output" sonrası parçayı yeniden bağla.
    parts_norm = [_norm_name(x) for x in p.parts]
    if "eye_roi_output" in parts_norm:
        idx = parts_norm.index("eye_roi_output")
        suffix_parts = p.parts[idx + 1:]
        if suffix_parts:
            candidates.append(EYE_ROI_ROOT.joinpath(*suffix_parts))

    # 5) Sadece dosya adı güvenilir kaldıysa canonical klasörü dene.
    candidates.append(EYE_ROI_ROOT / str(label) / str(split) / "combined" / p.name)

    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.is_file():
            return candidate

    return None

success_mask = metadata["status"].map(is_success_status)
eligible = metadata[
    success_mask
    & metadata["label"].isin(CONFIG["labels"])
    & metadata["split"].isin(CONFIG["splits"])
].copy()

if eligible.empty:
    raise RuntimeError("Metadata içinde eğitime uygun SUCCESS göz ROI kaydı bulunamadı.")

eligible["resolved_path"] = [
    resolve_combined_eye_path(row.combined_eye_path, row.label, row.split)
    for row in eligible.itertuples(index=False)
]

missing_files = eligible[eligible["resolved_path"].isna()].copy()
if len(missing_files):
    missing_files.to_csv(
        ARTIFACTS_DIR / "missing_combined_eye_files.csv",
        index=False,
        encoding="utf-8-sig",
    )
    examples = missing_files[["label", "split", "combined_eye_path"]].head(10)
    raise FileNotFoundError(
        f"{len(missing_files)} combined-eye ROI dosyası çözümlenemedi. "
        f"Örnekler:\n{examples.to_string(index=False)}\n"
        f"Tam liste: {ARTIFACTS_DIR / 'missing_combined_eye_files.csv'}"
    )

eligible["path"] = eligible["resolved_path"].map(str)
eligible["class_name"] = eligible["label"]
eligible["numeric_label"] = eligible["class_name"].map(CONFIG["labels"]).astype(np.int64)

# Metadata alanlarını mümkün olduğunca koru.
for optional_col in ["sample_id", "video_id", "source_video", "source_frame", "frame_stem", "face_id"]:
    if optional_col not in eligible.columns:
        eligible[optional_col] = ""

inventory_columns = [
    "path",
    "class_name",
    "numeric_label",
    "split",
    "sample_id",
    "video_id",
    "source_video",
    "source_frame",
    "frame_stem",
    "face_id",
]
inventory = eligible[inventory_columns].copy()
inventory = inventory.rename(columns={"numeric_label": "label"})

# Aynı fiziksel dosyanın tekrar kaydedilmesini engelle.
duplicate_paths = inventory[inventory.duplicated("path", keep=False)]
if len(duplicate_paths):
    duplicate_paths.to_csv(
        ARTIFACTS_DIR / "duplicate_paths.csv",
        index=False,
        encoding="utf-8-sig",
    )
    raise RuntimeError(
        f"Aynı combined-eye dosyası metadata içinde birden fazla kez bulundu: "
        f"{len(duplicate_paths)} kayıt."
    )

summary = (
    inventory
    .groupby(["split", "class_name"])
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "val", "test"], fill_value=0)
)

# Her splitte iki sınıf da zorunlu.
for split in CONFIG["splits"]:
    subset = inventory[inventory["split"] == split]
    if subset.empty:
        raise RuntimeError(f"{split} seti boş.")
    if set(subset["label"].unique()) != {0, 1}:
        raise RuntimeError(f"{split} setinde REAL ve FAKE sınıflarının ikisi de bulunmuyor.")

inventory.to_csv(
    ARTIFACTS_DIR / "dataset_inventory.csv",
    index=False,
    encoding="utf-8-sig",
)

display(summary)
print("\nToplam kullanılabilir combined-eye ROI:", len(inventory))

logger.info("Dataset inventory created. Total images=%d", len(inventory))
logger.info("\n%s", summary.to_string())


In [ ]:
# ============================================================
# 5) LEAKAGE / İÇERİK BÜTÜNLÜĞÜ KONTROLLERİ
# ============================================================

def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

hash_rows = []
for row in tqdm(inventory.itertuples(index=False), total=len(inventory), desc="SHA256 audit"):
    hash_rows.append({
        "path": row.path,
        "split": row.split,
        "class_name": row.class_name,
        "sha256": sha256_file(Path(row.path)),
    })

hash_df = pd.DataFrame(hash_rows)
hash_df.to_csv(
    ARTIFACTS_DIR / "image_sha256.csv",
    index=False,
    encoding="utf-8-sig",
)

cross_split_hashes = (
    hash_df.groupby("sha256")["split"]
    .nunique()
    .reset_index(name="split_count")
)
cross_split_hashes = cross_split_hashes[cross_split_hashes["split_count"] > 1]

if len(cross_split_hashes):
    bad_hashes = set(cross_split_hashes["sha256"])
    leakage_rows = hash_df[hash_df["sha256"].isin(bad_hashes)].copy()
    leakage_rows.to_csv(
        ARTIFACTS_DIR / "cross_split_content_leakage.csv",
        index=False,
        encoding="utf-8-sig",
    )
    raise RuntimeError(
        "Aynı görüntü içeriği birden fazla splitte bulundu. "
        "Deney güvenliği için işlem durduruldu."
    )

# video_id mevcut metadata'da gerçek source-video kimliği olmak zorunda değildir.
# Bu nedenle sadece raporlanır; güvenilir source_video yoksa video-level performans üretilmez.
def clean_id_series(series):
    s = series.fillna("").astype(str).str.strip()
    return s[~s.str.casefold().isin({"", "nan", "none", "null", "unknown", "n/a", "na"})]

source_video_available = False
source_video_leakage_status = "NOT_VERIFIABLE_FROM_CURRENT_METADATA"

if "source_video" in inventory.columns:
    valid_source_video = clean_id_series(inventory["source_video"])
    if len(valid_source_video) == len(inventory) and valid_source_video.nunique() > 2:
        tmp = inventory.copy()
        tmp["source_video"] = tmp["source_video"].astype(str).str.strip()
        source_video_split_counts = (
            tmp.groupby("source_video")["split"]
            .nunique()
            .reset_index(name="split_count")
        )
        leaks = source_video_split_counts[source_video_split_counts["split_count"] > 1]
        if len(leaks):
            leaks.to_csv(
                ARTIFACTS_DIR / "source_video_split_leakage.csv",
                index=False,
                encoding="utf-8-sig",
            )
            raise RuntimeError(
                "Güvenilir source_video alanında train/val/test leakage bulundu."
            )
        source_video_available = True
        source_video_leakage_status = "PASSED"

leakage_audit = {
    "duplicate_path_count": int(len(duplicate_paths)),
    "cross_split_duplicate_sha256_count": int(len(cross_split_hashes)),
    "source_video_available": bool(source_video_available),
    "source_video_split_leakage": source_video_leakage_status,
}

with open(ARTIFACTS_DIR / "leakage_audit.json", "w", encoding="utf-8") as f:
    json.dump(leakage_audit, f, indent=2, ensure_ascii=False)

print(json.dumps(leakage_audit, indent=2, ensure_ascii=False))
logger.info("Leakage audit: %s", leakage_audit)


In [ ]:
# ============================================================
# 6) VGG16 FEATURE EXTRACTOR
# ============================================================

feature_extractor = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
    pooling="avg",
)
feature_extractor.trainable = False

expected_dim = int(CONFIG["feature_extractor"]["feature_dim"])
actual_dim = int(feature_extractor.output_shape[-1])
if actual_dim != expected_dim:
    raise RuntimeError(
        f"Beklenen VGG16 feature boyutu {expected_dim}, gerçek boyut {actual_dim}."
    )

print("VGG16 output shape:", feature_extractor.output_shape)
print("Trainable:", feature_extractor.trainable)

with open(ARTIFACTS_DIR / "feature_extractor.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "architecture": "VGG16",
            "weights": "imagenet",
            "include_top": False,
            "pooling": "avg",
            "input_shape": [224, 224, 3],
            "output_dim": actual_dim,
            "trainable": False,
            "input_region": "combined_eye",
        },
        f,
        indent=2,
        ensure_ascii=False,
    )


In [ ]:
# ============================================================
# 7) GÖRÜNTÜ OKUMA + VGG16 FEATURE EXTRACTION + CACHE
# ============================================================

IMG_SIZE = tuple(CONFIG["image_size"])
BATCH_SIZE = int(CONFIG["batch_size"])

def load_batch(paths):
    batch = []
    valid_paths = []
    errors = []

    for p in paths:
        try:
            with Image.open(p) as img:
                img = img.convert("RGB")
                img = img.resize(IMG_SIZE, Image.Resampling.BILINEAR)
                arr = np.asarray(img, dtype=np.float32)

            if arr.shape != (IMG_SIZE[1], IMG_SIZE[0], 3):
                raise ValueError(f"Unexpected image shape: {arr.shape}")

            batch.append(arr)
            valid_paths.append(str(p))

        except (UnidentifiedImageError, OSError, ValueError) as e:
            errors.append((str(p), repr(e)))

    if not batch:
        return None, [], errors

    x = np.stack(batch, axis=0)
    x = preprocess_input(x)
    return x, valid_paths, errors

def current_split_signature(df):
    # Path + label + içerik hash'i cache geçerliliğine dahil edilir.
    split_hash_map = hash_df.set_index("path")["sha256"].to_dict()
    signature_payload = [
        (str(p), int(y), split_hash_map[str(p)])
        for p, y in zip(df["path"], df["label"])
    ]
    raw = json.dumps(signature_payload, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def extract_features_for_split(split_name):
    df = inventory[inventory["split"] == split_name].copy().reset_index(drop=True)
    if df.empty:
        raise RuntimeError(f"{split_name}: split boş.")

    cache_file = ARTIFACTS_DIR / f"{split_name}_features.npz"
    signature = current_split_signature(df)

    if CONFIG["reuse_cached_features"] and cache_file.exists():
        try:
            with np.load(cache_file, allow_pickle=True) as cached:
                cached_signature = str(cached["signature"].item())
                if cached_signature == signature:
                    logger.info("Cached %s features loaded.", split_name)
                    return {
                        "X": cached["features"].astype(np.float32),
                        "y": cached["labels"].astype(np.int64),
                        "paths": cached["paths"].astype(str).tolist(),
                        "sample_ids": cached["sample_ids"].astype(str).tolist(),
                        "video_ids": cached["video_ids"].astype(str).tolist(),
                        "source_videos": cached["source_videos"].astype(str).tolist(),
                        "class_names": cached["class_names"].astype(str).tolist(),
                    }
                logger.info("Cache signature mismatch for %s; recomputing.", split_name)
        except Exception as e:
            logger.warning("Cache read failed for %s (%r); recomputing.", split_name, e)

    all_features = []
    kept_indices = []
    bad_files = []

    for start in tqdm(
        range(0, len(df), BATCH_SIZE),
        desc=f"VGG16 features — {split_name}",
    ):
        chunk = df.iloc[start:start + BATCH_SIZE].copy()
        x, valid_paths, errors = load_batch(chunk["path"].tolist())
        bad_files.extend(errors)

        if x is None:
            continue

        feats = feature_extractor.predict(x, verbose=0)
        if feats.ndim != 2 or feats.shape[1] != 512:
            raise RuntimeError(f"{split_name}: beklenmeyen feature shape {feats.shape}")

        path_to_idx = {str(p): idx for idx, p in zip(chunk.index, chunk["path"])}
        valid_idx = [path_to_idx[p] for p in valid_paths]

        all_features.append(feats.astype(np.float32))
        kept_indices.extend(valid_idx)

    if bad_files:
        bad_df = pd.DataFrame(bad_files, columns=["path", "error"])
        bad_df.to_csv(
            ARTIFACTS_DIR / f"{split_name}_bad_files.csv",
            index=False,
            encoding="utf-8-sig",
        )
        raise RuntimeError(
            f"{split_name}: {len(bad_files)} görüntü okunamadı. "
            "Sessizce örnek düşürmemek için işlem durduruldu."
        )

    if not all_features:
        raise RuntimeError(f"{split_name}: Hiç geçerli görüntü işlenemedi.")

    kept = df.loc[kept_indices].reset_index(drop=True)
    X = np.concatenate(all_features, axis=0)
    y = kept["label"].astype(np.int64).to_numpy()

    if len(X) != len(kept) or len(y) != len(kept):
        raise RuntimeError(f"{split_name}: feature/metadata satır sayısı uyuşmuyor.")
    if not np.isfinite(X).all():
        raise RuntimeError(f"{split_name}: feature matrisinde NaN/Inf bulundu.")

    np.savez_compressed(
        cache_file,
        signature=np.asarray(signature),
        features=X,
        labels=y,
        paths=kept["path"].astype(str).to_numpy(dtype=object),
        sample_ids=kept["sample_id"].fillna("").astype(str).to_numpy(dtype=object),
        video_ids=kept["video_id"].fillna("").astype(str).to_numpy(dtype=object),
        source_videos=kept["source_video"].fillna("").astype(str).to_numpy(dtype=object),
        class_names=kept["class_name"].astype(str).to_numpy(dtype=object),
    )

    logger.info("%s features extracted: X=%s y=%s", split_name, X.shape, y.shape)

    return {
        "X": X,
        "y": y,
        "paths": kept["path"].astype(str).tolist(),
        "sample_ids": kept["sample_id"].fillna("").astype(str).tolist(),
        "video_ids": kept["video_id"].fillna("").astype(str).tolist(),
        "source_videos": kept["source_video"].fillna("").astype(str).tolist(),
        "class_names": kept["class_name"].astype(str).tolist(),
    }

train_data = extract_features_for_split("train")
val_data   = extract_features_for_split("val")
test_data  = extract_features_for_split("test")

X_train, y_train = train_data["X"], train_data["y"]
X_val,   y_val   = val_data["X"], val_data["y"]
X_test,  y_test  = test_data["X"], test_data["y"]

for name, X, y in [
    ("train", X_train, y_train),
    ("val", X_val, y_val),
    ("test", X_test, y_test),
]:
    if X.shape[0] != y.shape[0]:
        raise RuntimeError(f"{name}: X/y satır sayısı uyuşmuyor.")
    if X.shape[1] != 512:
        raise RuntimeError(f"{name}: feature_dim 512 değil: {X.shape}")
    if set(np.unique(y)) != {0, 1}:
        raise RuntimeError(f"{name}: iki sınıfın ikisi de yok.")

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)


In [ ]:
# ============================================================
# 8) SVM HİPERPARAMETRE SEÇİMİ — SADECE VALIDATION
# ============================================================

selection_scaler = StandardScaler()
X_train_scaled = selection_scaler.fit_transform(X_train)
X_val_scaled = selection_scaler.transform(X_val)

if not np.isfinite(X_train_scaled).all() or not np.isfinite(X_val_scaled).all():
    raise RuntimeError("StandardScaler sonrasında NaN/Inf bulundu.")

candidates = []

for C in CONFIG["svm"]["search_space"]["linear_C"]:
    candidates.append({"kernel": "linear", "C": float(C), "gamma": "scale"})

for C in CONFIG["svm"]["search_space"]["rbf_C"]:
    for gamma in CONFIG["svm"]["search_space"]["rbf_gamma"]:
        candidates.append({"kernel": "rbf", "C": float(C), "gamma": gamma})

search_rows = []

for params in tqdm(candidates, desc="SVM model selection"):
    model = SVC(
        kernel=params["kernel"],
        C=params["C"],
        gamma=params["gamma"],
        class_weight=CONFIG["svm"]["class_weight"],
        probability=False,
        random_state=SEED,
    )

    start = time.time()
    model.fit(X_train_scaled, y_train)
    elapsed = time.time() - start

    val_pred = model.predict(X_val_scaled)
    val_score = model.decision_function(X_val_scaled)

    row = {
        **params,
        "val_accuracy": float(accuracy_score(y_val, val_pred)),
        "val_precision": float(precision_score(y_val, val_pred, zero_division=0)),
        "val_recall": float(recall_score(y_val, val_pred, zero_division=0)),
        "val_f1": float(f1_score(y_val, val_pred, zero_division=0)),
        "val_roc_auc": float(roc_auc_score(y_val, val_score)),
        "fit_seconds": float(elapsed),
    }
    search_rows.append(row)

search_df = pd.DataFrame(search_rows).sort_values(
    ["val_f1", "val_roc_auc", "val_accuracy"],
    ascending=False,
).reset_index(drop=True)

if search_df.empty:
    raise RuntimeError("SVM model selection hiçbir sonuç üretmedi.")

search_df.to_csv(
    METRICS_DIR / "svm_validation_search.csv",
    index=False,
    encoding="utf-8-sig",
)

display(search_df)

best_gamma = search_df.loc[0, "gamma"]
if isinstance(best_gamma, str) and best_gamma not in {"scale", "auto"}:
    best_gamma = float(best_gamma)

best_params = {
    "kernel": str(search_df.loc[0, "kernel"]),
    "C": float(search_df.loc[0, "C"]),
    "gamma": best_gamma,
}

print("Best validation parameters:", best_params)
logger.info("Best validation parameters: %s", best_params)


In [ ]:
# ============================================================
# 9) FINAL MODEL — TRAIN + VAL ÜZERİNDE YENİDEN EĞİT
# ============================================================

X_trainval = np.concatenate([X_train, X_val], axis=0)
y_trainval = np.concatenate([y_train, y_val], axis=0)

final_scaler = StandardScaler()
X_trainval_scaled = final_scaler.fit_transform(X_trainval)
X_test_scaled = final_scaler.transform(X_test)

if not np.isfinite(X_trainval_scaled).all() or not np.isfinite(X_test_scaled).all():
    raise RuntimeError("Final scaling sonrasında NaN/Inf bulundu.")

final_svm = SVC(
    kernel=best_params["kernel"],
    C=best_params["C"],
    gamma=best_params["gamma"],
    class_weight=CONFIG["svm"]["class_weight"],
    probability=True,
    random_state=SEED,
)

start = time.time()
final_svm.fit(X_trainval_scaled, y_trainval)
final_fit_seconds = time.time() - start

if set(map(int, final_svm.classes_)) != {0, 1}:
    raise RuntimeError(f"Final SVM sınıfları beklenmiyor: {final_svm.classes_}")

joblib.dump(final_svm, CHECKPOINTS_DIR / "svm_model.joblib")
joblib.dump(final_scaler, CHECKPOINTS_DIR / "scaler.joblib")

# Fresh-load smoke test
reloaded_svm = joblib.load(CHECKPOINTS_DIR / "svm_model.joblib")
reloaded_scaler = joblib.load(CHECKPOINTS_DIR / "scaler.joblib")

smoke_n = min(8, len(X_test))
smoke_scaled = reloaded_scaler.transform(X_test[:smoke_n])
smoke_a = final_svm.predict(X_test_scaled[:smoke_n])
smoke_b = reloaded_svm.predict(smoke_scaled)

if not np.array_equal(smoke_a, smoke_b):
    raise RuntimeError("Fresh-load SVM inference smoke test başarısız.")

logger.info("Final SVM trained in %.2f seconds.", final_fit_seconds)
logger.info("Final model/scaler saved and fresh-load test passed.")


In [ ]:
# ============================================================
# 10) TEST TAHMİNLERİ + FRAME-LEVEL METRICS
# ============================================================

test_pred = final_svm.predict(X_test_scaled)
test_proba_all = final_svm.predict_proba(X_test_scaled)

class_to_col = {int(cls): i for i, cls in enumerate(final_svm.classes_)}
if 1 not in class_to_col:
    raise RuntimeError("FAKE=1 sınıfı final SVM classes_ içinde bulunamadı.")

fake_col = class_to_col[1]
test_fake_prob = test_proba_all[:, fake_col]
test_decision = final_svm.decision_function(X_test_scaled)

if not np.isfinite(test_fake_prob).all() or not np.isfinite(test_decision).all():
    raise RuntimeError("Test skorlarında NaN/Inf bulundu.")

frame_metrics = {
    "accuracy": float(accuracy_score(y_test, test_pred)),
    "precision": float(precision_score(y_test, test_pred, zero_division=0)),
    "recall": float(recall_score(y_test, test_pred, zero_division=0)),
    "f1_score": float(f1_score(y_test, test_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, test_fake_prob)),
    "average_precision": float(average_precision_score(y_test, test_fake_prob)),
    "n_test_frames": int(len(y_test)),
}

print(json.dumps(frame_metrics, indent=2, ensure_ascii=False))

report_dict = classification_report(
    y_test,
    test_pred,
    labels=[0, 1],
    target_names=["REAL", "FAKE"],
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).T
report_df.to_csv(
    METRICS_DIR / "classification_report.csv",
    encoding="utf-8-sig",
)

cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["true_REAL", "true_FAKE"],
    columns=["pred_REAL", "pred_FAKE"],
)
cm_df.to_csv(
    METRICS_DIR / "confusion_matrix.csv",
    encoding="utf-8-sig",
)

predictions_df = pd.DataFrame({
    "image_path": test_data["paths"],
    "sample_id": test_data["sample_ids"],
    "video_id": test_data["video_ids"],
    "source_video": test_data["source_videos"],
    "true_label": y_test,
    "true_class": ["FAKE" if x == 1 else "REAL" for x in y_test],
    "predicted_label": test_pred,
    "predicted_class": ["FAKE" if x == 1 else "REAL" for x in test_pred],
    "fake_probability": test_fake_prob,
    "decision_score": test_decision,
})

if len(predictions_df) != len(y_test):
    raise RuntimeError("Test prediction satır sayısı y_test ile uyuşmuyor.")

predictions_df.to_csv(
    PREDICTIONS_DIR / "test_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

display(report_df)
display(cm_df)


In [ ]:
# ============================================================
# 11) VIDEO-LEVEL DEĞERLENDİRME — YALNIZ GÜVENİLİRSE
# ============================================================

video_metrics = None
video_level_status = "NOT_AVAILABLE"

if source_video_available:
    video_df = (
        predictions_df
        .groupby(["true_label", "true_class", "source_video"], as_index=False)
        .agg(
            fake_probability=("fake_probability", "mean"),
            frame_count=("image_path", "size"),
        )
    )

    if video_df["true_label"].nunique() == 2 and len(video_df) < len(predictions_df):
        video_df["predicted_label"] = (video_df["fake_probability"] >= 0.5).astype(int)
        video_df["predicted_class"] = np.where(
            video_df["predicted_label"].eq(1),
            "FAKE",
            "REAL",
        )

        video_metrics = {
            "accuracy": float(accuracy_score(video_df["true_label"], video_df["predicted_label"])),
            "precision": float(precision_score(video_df["true_label"], video_df["predicted_label"], zero_division=0)),
            "recall": float(recall_score(video_df["true_label"], video_df["predicted_label"], zero_division=0)),
            "f1_score": float(f1_score(video_df["true_label"], video_df["predicted_label"], zero_division=0)),
            "roc_auc": float(roc_auc_score(video_df["true_label"], video_df["fake_probability"])),
            "n_test_videos": int(len(video_df)),
        }

        video_df.to_csv(
            PREDICTIONS_DIR / "video_level_predictions.csv",
            index=False,
            encoding="utf-8-sig",
        )
        video_level_status = "AVAILABLE"
        print("Video-level metrics:")
        print(json.dumps(video_metrics, indent=2, ensure_ascii=False))

if video_metrics is None:
    print(
        "Video-level metric üretilmedi. Mevcut metadata içindeki video_id alanı "
        "orijinal kaynak-video kimliği olarak doğrulanamadığı için bilimsel olarak "
        "yanlış bir video-level başarı raporu oluşturulmuyor."
    )


In [ ]:
# ============================================================
# 12) FIGURES
# ============================================================

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111)
ax.imshow(cm)
ax.set_title("Confusion Matrix — Eye ROI VGG16 Features + SVM")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_xticks([0, 1], labels=["REAL", "FAKE"])
ax.set_yticks([0, 1], labels=["REAL", "FAKE"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

fpr, tpr, _ = roc_curve(y_test, test_fake_prob)
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111)
ax.plot(fpr, tpr, label=f"AUC = {frame_metrics['roc_auc']:.4f}")
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_title("ROC Curve — Eye ROI VGG16 Features + SVM")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "roc_curve.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

precision_curve, recall_curve, _ = precision_recall_curve(y_test, test_fake_prob)
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111)
ax.plot(
    recall_curve,
    precision_curve,
    label=f"AP = {frame_metrics['average_precision']:.4f}",
)
ax.set_title("Precision-Recall Curve — Eye ROI VGG16 Features + SVM")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "precision_recall_curve.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)


In [ ]:
# ============================================================
# 13) METRICS / ENVIRONMENT / SUMMARY DOSYALARI
# ============================================================

metrics_payload = {
    "experiment": RUN_NAME,
    "region": "eye",
    "roi_variant": "combined",
    "feature_extractor": "VGG16 ImageNet / include_top=False / pooling=avg",
    "classifier": "SVM",
    "best_svm_params": best_params,
    "frame_level": frame_metrics,
    "video_level_status": video_level_status,
    "video_level": video_metrics,
    "leakage_audit": leakage_audit,
}

with open(METRICS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2, ensure_ascii=False)

gpu_devices = tf.config.list_physical_devices("GPU")
environment_payload = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "tensorflow": tf.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": __import__("sklearn").__version__,
    "gpu_devices": [str(x) for x in gpu_devices],
    "seed": SEED,
}

with open(OUTPUT_DIR / "environment.json", "w", encoding="utf-8") as f:
    json.dump(environment_payload, f, indent=2, ensure_ascii=False)

try:
    requirements = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
    )
except Exception as e:
    requirements = f"# pip freeze failed: {repr(e)}\n"

with open(OUTPUT_DIR / "requirements_lock.txt", "w", encoding="utf-8") as f:
    f.write(requirements)

run_summary = {
    "status": "completed",
    "experiment_name": RUN_NAME,
    "region": "eye",
    "roi_variant": "combined",
    "dataset_dir": str(EYE_ROI_ROOT),
    "metadata_csv": str(METADATA_CSV),
    "output_dir": str(OUTPUT_DIR),
    "dataset_counts": (
        inventory.groupby(["split", "class_name"])
        .size()
        .rename("count")
        .reset_index()
        .to_dict(orient="records")
    ),
    "feature_shapes": {
        "train": list(X_train.shape),
        "val": list(X_val.shape),
        "test": list(X_test.shape),
    },
    "best_svm_params": best_params,
    "final_fit_seconds": float(final_fit_seconds),
    "frame_level_metrics": frame_metrics,
    "video_level_status": video_level_status,
    "video_level_metrics": video_metrics,
    "leakage_audit": leakage_audit,
}

with open(OUTPUT_DIR / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2, ensure_ascii=False)

logger.info("Metrics and run metadata saved.")


In [ ]:
# ============================================================
# 14) OUTPUT MANIFEST
# ============================================================

manifest_rows = []

for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file() and p.name != "output_manifest.csv":
        manifest_rows.append({
            "relative_path": str(p.relative_to(OUTPUT_DIR)),
            "size_bytes": int(p.stat().st_size),
            "sha256": sha256_file(p),
        })

manifest_df = pd.DataFrame(
    manifest_rows,
    columns=["relative_path", "size_bytes", "sha256"],
)
manifest_df.to_csv(
    OUTPUT_DIR / "output_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

display(manifest_df)

logger.info("Output manifest saved. Files=%d", len(manifest_df))
print("\n✅ DENEY TAMAMLANDI")
print("Sonuç klasörü:", OUTPUT_DIR)


## Tamamlandı

Bu sürüm kaş deneyinin **VGG16 ImageNet feature extractor + StandardScaler + SVM**
mantığını korur; yalnızca veri tarafını Kader'in **combined-eye ROI** yapısına güvenli
biçimde uyarlamıştır.

Önemli farklar:
- `Göz/eye_roi_output/metadata.csv` kullanılır.
- Yalnız `SUCCESS` combined-eye ROI örnekleri eğitime alınır.
- Mevcut train/val/test splitleri değiştirilmez.
- Exact-content SHA-256 cross-split leakage kontrolü yapılır.
- Metadata'daki `video_id` güvenilir source-video kimliği değilse video-level metrik üretilmez.
- Feature cache geçerliliği path + label + içerik SHA-256 imzasıyla kontrol edilir.
- Final SVM/scaler kaydından sonra fresh-load inference smoke test yapılır.
